In [1]:
from bayes import *



In [11]:
train_docs = [
        "win free money now",
        "free lottery ticket winner",
        "claim your prize today free",
        "urgent offer free cash",
        "congratulations you won free",
        "meeting tomorrow at noon",
        "project update attached",
        "can we schedule a call",
        "quarterly report review",
        "lunch on thursday sounds good",
        "team standup notes attached",
        "please review the pull request",
    ]

train_labels = [
    "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham", "ham",
]

classifier = NaiveBayes(smoothing=0.1)
classifier.train(train_docs, train_labels)

print(f"\n  Training: {len(train_docs)} documents ({sum(1 for l in train_labels if l == 'spam')} spam, {sum(1 for l in train_labels if l == 'ham')} ham)")
print(f"  Vocabulary size: {len(classifier.vocab)}")

test_messages = [
    "free money waiting for you",
    "meeting rescheduled to friday",
    "you won a free prize",
    "please review the attached report",
    "urgent free offer claim now",
    "can we discuss the project update",
]

print("\n  Predictions:")
for msg in test_messages:
    prediction = classifier.predict(msg)
    proba = classifier.predict_proba(msg)
    confidence = proba[prediction]
    print(f"    '{msg}'")
    print(f"      -> {prediction} (confidence: {confidence:.3f})")

    print("\n  Top 5 spam indicator words:")
    for word, prob in classifier.top_words("spam", 5):
        print(f"    {word}: {prob:.4f}")

    print("\n  Top 5 ham indicator words:")
    for word, prob in classifier.top_words("ham", 5):
        print(f"    {word}: {prob:.4f}")


  Training: 12 documents (5 spam, 7 ham)
  Vocabulary size: 44

  Predictions:
    'free money waiting for you'
      -> spam (confidence: 1.000)

  Top 5 spam indicator words:
    free: 0.2008
    winner: 0.0433
    win: 0.0433
    your: 0.0433
    now: 0.0433

  Top 5 ham indicator words:
    review: 0.0629
    attached: 0.0629
    call: 0.0329
    can: 0.0329
    lunch: 0.0329
    'meeting rescheduled to friday'
      -> ham (confidence: 0.837)

  Top 5 spam indicator words:
    free: 0.2008
    winner: 0.0433
    win: 0.0433
    your: 0.0433
    now: 0.0433

  Top 5 ham indicator words:
    review: 0.0629
    attached: 0.0629
    call: 0.0329
    can: 0.0329
    lunch: 0.0329
    'you won a free prize'
      -> spam (confidence: 1.000)

  Top 5 spam indicator words:
    free: 0.2008
    winner: 0.0433
    win: 0.0433
    your: 0.0433
    now: 0.0433

  Top 5 ham indicator words:
    review: 0.0629
    attached: 0.0629
    call: 0.0329
    can: 0.0329
    lunch: 0.0329
    'please 

# Exercice 3 

In [8]:
from collections import defaultdict
import math
class NaiveBayesLength:
    def __init__(self, smoothing=1.0,length_threshold=10):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()
        self.length_threshold = length_threshold
        self.class_length = defaultdict(lambda : defaultdict(int))

    def train(self, documents, labels):
        for doc, label in zip(documents, labels):
            self.class_counts[label] += 1
            words = doc.lower().split()
            if len(words) > self.length_threshold :
                self.class_length[label]["long"] += 1
            else : 
                self.class_length[label]["short"] += 1
            for word in words:
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def _log_prior(self, cls):
        total_docs = sum(self.class_counts.values())
        return math.log(self.class_counts[cls] / total_docs)

    def _log_likelihood(self, word, cls):
        count = self.word_counts[cls].get(word, 0)
        total = self.class_word_totals[cls]
        vocab_size = len(self.vocab)
        return math.log(
            (count + self.smoothing) / (total + self.smoothing * vocab_size)
        )

    def _log_length_likelihood(self,length_type,cls) : 
        length_count = self.class_length[cls][length_type]
        total_docs = self.class_counts[cls]
        p_length = (length_count+self.smoothing)/ (total_docs +2 * self.smoothing)
        return math.log(p_length)

    def predict(self, document):
        words = document.lower().split()
        best_class = None
        best_score = float("-inf")
        length = len(words)
        if length > self.length_threshold :
            length_type = "long"
        else : 
            length_type ="short"
        #compute P(L|c)


        for cls in self.class_counts:
            score = self._log_prior(cls)
            for word in words:
                score += self._log_likelihood(word, cls)
            score += self._log_length_likelihood(length_type,cls)
            if score > best_score:
                best_score = score
                best_class = cls

        return best_class

    def predict_proba(self, document):
        words = document.lower().split()
        scores = {}

        if len(words) > self.length_threshold :
            length_type = "long"
        else : 
            length_type ="short"
        for cls in self.class_counts:
            score = self._log_prior(cls)
            for word in words:
                score += self._log_likelihood(word, cls)
            score += self._log_length_likelihood(length_type,cls)
            scores[cls] = score

        max_score = max(scores.values())
        exp_scores = {cls: math.exp(s - max_score) for cls, s in scores.items()}
        total = sum(exp_scores.values())
        return {cls: exp_scores[cls] / total for cls in exp_scores}

    def top_words(self, cls, n=10):
        vocab_size = len(self.vocab)
        total = self.class_word_totals[cls]
        probs = {}
        for word in self.vocab:
            count = self.word_counts[cls].get(word, 0)
            probs[word] = (count + self.smoothing) / (
                total + self.smoothing * vocab_size
            )
        return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:n]

In [9]:
classifier = NaiveBayesLength(smoothing=0.1)
documents = [
    # Spam courts
    "win money now",
    "free prize",
    "claim your reward",
    "limited offer",
    "win cash today",

    # Spam longs
    "congratulations you have won a free prize claim your money now",
    "exclusive offer available today click here to receive your free reward",
    "you have been selected to win money and receive an amazing gift",

    # Ham courts
    "hello how are you",
    "see you tomorrow",
    "meeting at noon",
    "thanks for your help",
    "call me later",

    # Ham longs
    "hello can we schedule a meeting tomorrow morning to discuss the project",
    "please review the document I sent yesterday and give me your feedback",
    "we need to organize a meeting with the team next week"
]

labels = ["spam"] * 8 + ["ham"] * 8

classifier.train(documents,labels)

print("Nombre de documents :", dict(classifier.class_counts))

for cls in classifier.class_counts:
    print(f"\nClasse : {cls}")
    print("Longueurs :", dict(classifier.class_length[cls]))

Nombre de documents : {'spam': 8, 'ham': 8}

Classe : spam
Longueurs : {'short': 5, 'long': 3}

Classe : ham
Longueurs : {'short': 5, 'long': 3}


In [10]:
test_messages = [
    "win money",
    "hello friend",
    "claim your free money now",
    "hello can we schedule a meeting tomorrow morning to discuss our project",
    "you have won an exclusive free prize claim your reward today"
]

for message in test_messages:
    prediction = classifier.predict(message)
    probabilities = classifier.predict_proba(message)

    print(f"\nMessage : {message}")
    print(f"Longueur : {len(message.split())} mots")
    print(f"Prédiction : {prediction}")

    for cls, prob in probabilities.items():
        print(f"P({cls} | message) = {prob:.2%}")


Message : win money
Longueur : 2 mots
Prédiction : spam
P(spam | message) = 99.91%
P(ham | message) = 0.09%

Message : hello friend
Longueur : 2 mots
Prédiction : ham
P(spam | message) = 5.39%
P(ham | message) = 94.61%

Message : claim your free money now
Longueur : 5 mots
Prédiction : spam
P(spam | message) = 100.00%
P(ham | message) = 0.00%

Message : hello can we schedule a meeting tomorrow morning to discuss our project
Longueur : 12 mots
Prédiction : ham
P(spam | message) = 0.00%
P(ham | message) = 100.00%

Message : you have won an exclusive free prize claim your reward today
Longueur : 11 mots
Prédiction : spam
P(spam | message) = 100.00%
P(ham | message) = 0.00%
